<div dir="rtl" align="right">

# פרויקט סיום — ניתוח נתוני סוכרת

**מחבר:** רון ברמי | **GitHub:** [rdiol12](https://github.com/rdiol12)

---

## תיאור הדאטהסט

הקובץ מכיל 4,522 רשומות של מטופלות מחמש מרפאות, בין השנים 2022–2024.
לכל מטופלת נמדדו מדדים רפואיים (גלוקוז, לחץ דם, BMI, HbA1c, אינסולין),
נתוני רקע (גיל, מספר היריונות, היסטוריה משפחתית) והרגלים
(פעילות גופנית, עישון). העמודה `Diagnosis` מציינת אם המטופלת אובחנה
כחולת סוכרת.

מטרת העבודה: לנקות את הנתונים, לאפיין את האוכלוסייה, ולבדוק אילו גורמים
קשורים לתחלואה בסוכרת.

## מבנה העבודה

1. עיבוד מקדים של הנתונים
2. סקירה כללית
3. ניתוח מעמיק — השוואה בין שתי אוכלוסיות
4. בדיקות סטטיסטיות
5. סיכום ומסקנות

**הערה:** יש להריץ את המחברת מלמעלה למטה (Run All). הקובץ
`diabetes_final_project.csv` צריך להימצא באותה תיקייה.

</div>

<div dir="rtl" align="right">

---
# 1. עיבוד מקדים

## 1.1 ייבוא ספריות וטעינת הנתונים

הנתונים נטענים כטקסט (`dtype=str`) כדי לראות את הערכים כמו שהם ולזהות
בעיות לפני ההמרה לטיפוסים מספריים.

</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
df = pd.read_csv("diabetes_final_project.csv", dtype=str)
print("מספר שורות:", df.shape[0])
print("מספר עמודות:", df.shape[1])
df.head()

In [ ]:
df.info()

<div dir="rtl" align="right">

## 1.2 בדיקת ערכים חסרים

</div>

In [ ]:
missing = pd.DataFrame({
    "חסרים": df.isna().sum(),
    "אחוז": (df.isna().mean() * 100).round(2)
})
missing[missing["חסרים"] > 0].sort_values("חסרים", ascending=False)

<div dir="rtl" align="right">

שש עמודות מכילות ערכים חסרים. הגדולות שבהן הן `BMI` (218) ו-`HbA1c` (180).
נטפל בהן בהמשך אחרי המרת העמודות לטיפוס מספרי.

</div>

<div dir="rtl" align="right">

## 1.3 בדיקת ערכים לא אחידים

בעמודות הטקסט אותו ערך מופיע לפעמים באותיות גדולות ולפעמים בקטנות.
מבחינת pandas אלה ערכים שונים לחלוטין, ולכן צריך לאחד אותם.

</div>

In [ ]:
text_cols = ["Clinic", "Gender", "BPCategory", "FamilyHistory",
             "PhysicalActivity", "SmokingStatus", "Diagnosis"]

for c in text_cols:
    print(c, ":", sorted(df[c].dropna().unique()))

<div dir="rtl" align="right">

אפשר לראות למשל ש-`FamilyHistory` מכילה `No`, `no`, `Yes`, `YES` —
ארבעה ערכים שהם למעשה שניים. אותה בעיה קיימת גם ב-`PhysicalActivity`,
`SmokingStatus` ו-`Diagnosis`.

</div>

<div dir="rtl" align="right">

## 1.4 טיפול בעמודות טקסט

מורידים רווחים מיותרים והופכים הכל לאותיות קטנות.

</div>

In [ ]:
for c in text_cols:
    df[c] = df[c].str.strip().str.lower()

for c in text_cols:
    print(c, ":", sorted(df[c].dropna().unique()))

<div dir="rtl" align="right">

## 1.5 עיבוד תאריכים

בעמודת `DiagnosisDate` מופיעים שלושה פורמטים שונים של תאריך:

* `2022-04-19`
* `26/07/2022`
* `Feb 24, 2023`

כתבתי פונקציה שמנסה את שלושת הפורמטים לפי הסדר. הפורמט עם הלוכסנים הוא
יום/חודש/שנה — בדקתי וראיתי שהחלק הראשון מגיע עד 27, כלומר הוא לא יכול
להיות מספר החודש.

</div>

In [ ]:
print(df.DiagnosisDate.sample(8, random_state=1).tolist())

In [ ]:
def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    value = value.strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y"):
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["DiagnosisDate"] = df["DiagnosisDate"].apply(parse_date)

print("תאריכים שלא הומרו:", df.DiagnosisDate.isna().sum())
print("טווח התאריכים:", df.DiagnosisDate.min().date(), "עד", df.DiagnosisDate.max().date())

In [ ]:
# עמודות עזר לניתוח לפי זמן
df["DiagnosisYear"] = df.DiagnosisDate.dt.year
df["DiagnosisQuarter"] = df.DiagnosisDate.dt.to_period("Q").astype(str)

df[["DiagnosisDate", "DiagnosisYear", "DiagnosisQuarter"]].head()

<div dir="rtl" align="right">

## 1.6 הסרת רשומות כפולות

בודקים כפילויות רק אחרי שאיחדנו את הטקסט והתאריכים, אחרת אותה רשומה
בפורמט שונה לא תיספר ככפילות.

</div>

In [ ]:
print("רשומות כפולות:", df.PatientID.duplicated().sum())

df = df.drop_duplicates(subset=["PatientID"], keep="first").reset_index(drop=True)
print("שורות אחרי ההסרה:", len(df))

<div dir="rtl" align="right">

## 1.7 המרת עמודות למספרים

בעמודות המספריות יש שלוש בעיות שמונעות המרה ישירה:

| עמודה | הבעיה | דוגמה |
|---|---|---|
| `GlucoseMgDl` | יחידות בתוך הערך | `110 mg/dL` |
| `HbA1c` | סימן אחוז | `5.4%` |
| `BMI` | פסיק במקום נקודה עשרונית | `25,1` |

</div>

In [ ]:
num_cols = ["Age", "Pregnancies", "GlucoseMgDl", "GlucoseFollowUp",
            "BloodPressureMmHg", "SkinThicknessMm", "InsulinUuMl",
            "BMI", "HbA1c", "PedigreeScore"]

for c in num_cols:
    df[c] = (df[c].astype(str)
                  .str.strip()
                  .str.replace("mg/dL", "", regex=False)
                  .str.replace("%", "", regex=False)
                  .str.replace(",", ".", regex=False)
                  .replace({"nan": np.nan, "": np.nan}))
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[num_cols].describe().round(2)

<div dir="rtl" align="right">

## 1.8 תיקון שגיאות

בטבלה למעלה אפשר לראות ערכים שלא הגיוניים מבחינה רפואית: גיל מינימלי
שלילי, BMI מקסימלי של 250, גלוקוז של 999. אלה שגיאות קלט.

הגדרתי טווח תקין לכל עמודה. ערך שנמצא מחוץ לטווח מסומן כערך חסר —
לא מוחקים את כל השורה בגלל תא אחד שגוי.

</div>

In [ ]:
valid_range = {
    "Age": (10, 120),
    "Pregnancies": (0, 20),
    "GlucoseMgDl": (30, 400),
    "GlucoseFollowUp": (30, 400),
    "BloodPressureMmHg": (40, 200),
    "SkinThicknessMm": (5, 100),
    "InsulinUuMl": (1, 900),
    "BMI": (12, 70),
    "HbA1c": (3, 20),
    "PedigreeScore": (0, 3),
}

for c, (low_limit, high_limit) in valid_range.items():
    bad = df[c].notna() & ~df[c].between(low_limit, high_limit)
    if bad.sum() > 0:
        print(f"{c}: {bad.sum()} ערכים שגויים -> {sorted(df.loc[bad, c].unique())[:4]}")
        df.loc[bad, c] = np.nan

<div dir="rtl" align="right">

## 1.9 השלמת ערכים חסרים

**עמודות מספריות** — השלמה בחציון של קבוצת האבחון שאליה שייכת המטופלת,
ולא בחציון של כל הטבלה. אם משלימים בחציון הכללי, הערכים של הקבוצות
מתקרבים זה לזה וההבדל ביניהן נעלם. בחרתי חציון ולא ממוצע כי הוא פחות
מושפע מערכים קיצוניים.

**עמודות טקסט** — ערך `unknown`, כי אי אפשר לנחש את הערך האמיתי.

</div>

In [ ]:
for c in num_cols:
    n = df[c].isna().sum()
    if n > 0:
        df[c] = df.groupby("Diagnosis")[c].transform(lambda s: s.fillna(s.median()))
        print(f"{c}: הושלמו {n} ערכים")

for c in ["PhysicalActivity", "SmokingStatus"]:
    n = df[c].isna().sum()
    if n > 0:
        df[c] = df[c].fillna("unknown")
        print(f"{c}: {n} ערכים סומנו כ-unknown")

print("\nסך ערכים חסרים שנותרו:", df.isna().sum().sum())

<div dir="rtl" align="right">

## 1.10 טיפוסים סופיים ועמודות חדשות

בעמודות כמו גיל ולחץ דם אין משמעות לשבר עשרוני, ולכן הן מומרות ל-int.

הוספתי ארבע עמודות שיעזרו בניתוח:

* `AgeGroup` — קיבוץ הגילאים לקבוצות
* `BMICategory` — סיווג לפי טבלת ה-BMI המקובלת
* `GlucoseDelta` — ההפרש בין מדידת המעקב למדידה הראשונה
* `IsDiabetic` — עמודה בינארית (0/1) לצורך חישובי אחוזים

</div>

In [ ]:
int_cols = ["Age", "Pregnancies", "GlucoseMgDl", "GlucoseFollowUp",
            "BloodPressureMmHg", "InsulinUuMl"]
for c in int_cols:
    df[c] = df[c].round().astype(int)

# חשוב להגדיר את הקטגוריות כמסודרות, אחרת הגרפים יסודרו לפי סדר אלפביתי
df["AgeGroup"] = pd.cut(df.Age, [0, 30, 40, 50, 60, 120],
                        labels=["<30", "30-39", "40-49", "50-59", "60+"])
df["BMICategory"] = pd.cut(df.BMI, [0, 18.5, 25, 30, 100],
                           labels=["Underweight", "Normal", "Overweight", "Obese"])
df["GlucoseDelta"] = df.GlucoseFollowUp - df.GlucoseMgDl
df["IsDiabetic"] = (df.Diagnosis == "diabetic").astype(int)

df.dtypes

<div dir="rtl" align="right">

עמודת `Gender` מכילה ערך אחד בלבד בכל השורות, ולכן אין בה שום מידע
שימושי לניתוח. מסירים אותה.

</div>

In [ ]:
print("ערכים בעמודת Gender:", df.Gender.unique())
df = df.drop(columns=["Gender"])

<div dir="rtl" align="right">

## 1.11 בדיקה סופית ושמירה

</div>

In [ ]:
print("ערכים חסרים:", df.isna().sum().sum())
print("כפילויות:", df.PatientID.duplicated().sum())
print("תאריכים לא תקינים:", df.DiagnosisDate.isna().sum())
print("שיעור החולות:", round(df.IsDiabetic.mean() * 100, 1), "%")
print("\nגודל הטבלה הנקייה:", df.shape)

df.to_csv("diabetes_clean.csv", index=False, encoding="utf-8-sig")

<div dir="rtl" align="right">

---
# 2. סקירה כללית

## 2.1 ספירת ערכים

</div>

In [ ]:
cat_cols = ["Diagnosis", "Clinic", "BPCategory", "FamilyHistory",
            "PhysicalActivity", "SmokingStatus", "AgeGroup", "BMICategory"]

for c in cat_cols:
    counts = df[c].value_counts()
    percent = (counts / len(df) * 100).round(1)
    print(f"--- {c} ---")
    print(pd.DataFrame({"כמות": counts, "אחוז": percent}))
    print()

In [ ]:
df[num_cols].describe().T.round(2)

<div dir="rtl" align="right">

## 2.2 פעולות GroupBy

### פרופיל ממוצע לפי אבחון

</div>

In [ ]:
df.groupby("Diagnosis")[num_cols].mean().round(2)

<div dir="rtl" align="right">

ההבדלים הבולטים: גלוקוז 142 אצל החולות מול 111 אצל הבריאות,
ו-BMI של 36.6 מול 30.1.

</div>

<div dir="rtl" align="right">

### תחלואה לפי פעילות גופנית

</div>

In [ ]:
activity_order = ["low", "moderate", "high", "unknown"]

df.groupby("PhysicalActivity").agg(
    כמות=("PatientID", "count"),
    אחוז_חולות=("IsDiabetic", lambda s: round(s.mean() * 100, 1)),
    ממוצע_BMI=("BMI", "mean"),
    ממוצע_גלוקוז=("GlucoseMgDl", "mean")
).reindex(activity_order).round(1)

<div dir="rtl" align="right">

### קבוצת גיל לפי אבחון

</div>

In [ ]:
df.groupby(["AgeGroup", "Diagnosis"], observed=True).agg(
    כמות=("PatientID", "count"),
    ממוצע_גלוקוז=("GlucoseMgDl", "mean"),
    ממוצע_BMI=("BMI", "mean")
).round(1)

<div dir="rtl" align="right">

### השוואה בין המרפאות

</div>

In [ ]:
df.groupby("Clinic").agg(
    כמות=("PatientID", "count"),
    אחוז_חולות=("IsDiabetic", lambda s: round(s.mean() * 100, 1)),
    ממוצע_גיל=("Age", "mean"),
    ממוצע_גלוקוז=("GlucoseMgDl", "mean")
).round(1)

<div dir="rtl" align="right">

### מגמה לאורך זמן

</div>

In [ ]:
df.groupby("DiagnosisQuarter").agg(
    כמות=("PatientID", "count"),
    אחוז_חולות=("IsDiabetic", lambda s: round(s.mean() * 100, 1))
)

<div dir="rtl" align="right">

### היסטוריה משפחתית מול פעילות גופנית

</div>

In [ ]:
(pd.crosstab(df.FamilyHistory, df.PhysicalActivity,
             values=df.IsDiabetic, aggfunc="mean") * 100).round(1)

<div dir="rtl" align="right">

מהחיתוכים עולה:

* שיעור התחלואה בקבוצת הפעילות הנמוכה גבוה פי שניים מזה שבקבוצת
  הפעילות הגבוהה (44% מול 21%).
* חמש המרפאות דומות מאוד זו לזו (34%–36%).
* אין מגמה לאורך זמן — האחוזים נעים סביב אותו ערך בכל הרבעונים.
* גם למי שיש היסטוריה משפחתית, פעילות גופנית גבוהה קשורה לשיעור
  תחלואה נמוך יותר (47% מול 23%).

</div>

<div dir="rtl" align="right">

## 2.3 ויזואליזציות

</div>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.countplot(data=df, x="Diagnosis", ax=axes[0, 0])
axes[0, 0].set_title("התפלגות האבחון"[::-1])

sns.countplot(data=df, x="PhysicalActivity", hue="Diagnosis",
              order=activity_order, ax=axes[0, 1])
axes[0, 1].set_title("Physical Activity by Diagnosis")

sns.countplot(data=df, x="AgeGroup", hue="Diagnosis", ax=axes[1, 0])
axes[1, 0].set_title("Age Group by Diagnosis")

sns.countplot(data=df, x="BMICategory", hue="Diagnosis", ax=axes[1, 1])
axes[1, 1].set_title("BMI Category by Diagnosis")

axes[0, 0].set_title("Diagnosis Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.flat, ["GlucoseMgDl", "HbA1c", "BMI",
                             "InsulinUuMl", "Age", "BloodPressureMmHg"]):
    sns.histplot(data=df, x=c, hue="Diagnosis", kde=True, element="step", ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, c in zip(axes, ["GlucoseMgDl", "HbA1c", "BMI", "InsulinUuMl"]):
    sns.boxplot(data=df, x="Diagnosis", y=c, ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

(df.groupby("PhysicalActivity").IsDiabetic.mean() * 100).reindex(
    activity_order).plot(kind="bar", ax=axes[0], color="#e07a5f")
axes[0].set_title("Diabetic Rate by Physical Activity (%)")

(df.groupby("AgeGroup", observed=True).IsDiabetic.mean() * 100).plot(
    kind="bar", ax=axes[1], color="#3d5a80")
axes[1].set_title("Diabetic Rate by Age Group (%)")

(df.groupby("BMICategory", observed=True).IsDiabetic.mean() * 100).plot(
    kind="bar", ax=axes[2], color="#81b29a")
axes[2].set_title("Diabetic Rate by BMI Category (%)")

for ax in axes:
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
(df.groupby("DiagnosisQuarter").IsDiabetic.mean() * 100).plot(
    marker="o", ax=ax, color="#3d5a80")
ax.set_title("Diabetic Rate by Quarter (%)")
ax.set_ylabel("%")
ax.set_ylim(0, 60)
plt.tight_layout()
plt.show()

<div dir="rtl" align="right">

## 2.4 מטריצת קורלציות

</div>

In [ ]:
corr = df[num_cols + ["IsDiabetic"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
corr["IsDiabetic"].drop("IsDiabetic").sort_values(ascending=False).round(3)

<div dir="rtl" align="right">

המשתנים בעלי הקשר החזק ביותר לאבחון הם הגלוקוז (0.49) וה-HbA1c (0.48),
ואחריהם האינסולין (0.39) וה-BMI (0.33).

בנוסף אפשר לראות במטריצה קורלציה גבוהה מאוד בין `HbA1c` ל-`GlucoseMgDl`
(0.90) ובין `GlucoseFollowUp` ל-`GlucoseMgDl` (0.98). זה הגיוני, כי כל
השלושה מודדים את אותו דבר.

</div>

<div dir="rtl" align="right">

## 2.5 חלוקה לקבוצות בעזרת K-Means

לפני ההרצה מבצעים סטנדרטיזציה של הנתונים. K-Means מחשב מרחקים, ובלי
סטנדרטיזציה עמודה כמו `InsulinUuMl` (ערכים של מאות) הייתה משפיעה על
התוצאה הרבה יותר מ-`HbA1c` (ערכים חד-ספרתיים).

עמודת `IsDiabetic` לא נכללת — האלגוריתם מחלק לקבוצות בלי לדעת מי חולה.

</div>

In [ ]:
features = ["Age", "GlucoseMgDl", "BMI", "HbA1c", "InsulinUuMl",
            "BloodPressureMmHg", "PedigreeScore", "Pregnancies"]

X = StandardScaler().fit_transform(df[features])
print("גודל מטריצת הנתונים:", X.shape)

<div dir="rtl" align="right">

### בחירת מספר הקבוצות

בודקים שני מדדים: אינרציה (שיטת המרפק) וציון Silhouette.

</div>

In [ ]:
results = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    results.append({
        "k": k,
        "inertia": round(model.inertia_, 1),
        "silhouette": round(silhouette_score(X, model.labels_), 3)
    })

results = pd.DataFrame(results).set_index("k")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
results.inertia.plot(marker="o", ax=axes[0], title="Elbow Method")
results.silhouette.plot(marker="o", ax=axes[1], color="#e07a5f",
                        title="Silhouette Score")
plt.tight_layout()
plt.show()

results

<div dir="rtl" align="right">

בחרתי ב-k=3. ציון ה-Silhouette הגבוה ביותר הוא אמנם ב-k=2, אבל חלוקה
לשתי קבוצות רק חוזרת על החלוקה לחולות ובריאות שכבר קיימת בנתונים.
בגרף המרפק רואים ירידה חדה עד k=3 ואחר כך התמתנות, ולכן שלוש קבוצות
נותנות חלוקה מפורטת יותר בלי לפצל יתר על המידה.

</div>

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
df["Cluster"] = kmeans.labels_

df.groupby("Cluster").agg(
    כמות=("PatientID", "count"),
    **{c: (c, "mean") for c in features},
    אחוז_חולות=("IsDiabetic", lambda s: round(s.mean() * 100, 1))
).round(1)

In [ ]:
pd.crosstab(df.Cluster, df.Diagnosis, normalize="index").mul(100).round(1)

In [ ]:
pca = PCA(n_components=2, random_state=42).fit(X)
coords = pca.transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df.Cluster,
                palette="Set1", alpha=0.6, s=18, ax=axes[0])
axes[0].set_title("K-Means Clusters (k=3)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

cluster_means = df.groupby("Cluster")[features].mean()
normalized = (cluster_means - df[features].mean()) / df[features].std()
sns.heatmap(normalized.T, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=axes[1])
axes[1].set_title("Cluster Profiles")

plt.tight_layout()
plt.show()

<div dir="rtl" align="right">

שלוש הקבוצות שהתקבלו:

| קבוצה | כמות | מאפיינים | אחוז חולות |
|---|---|---|---|
| סיכון נמוך | 2,322 | ממוצע גיל 26, כל המדדים מתחת לממוצע | 14.4% |
| סיכון בינוני | 1,187 | ממוצע גיל 46, 7.4 היריונות, מדדים רפואיים תקינים | 43.7% |
| סיכון גבוה | 991 | גלוקוז 162, HbA1c 6.1, BMI 37 | 73.1% |

הקבוצה האמצעית מעניינת במיוחד: אחוז החולות בה גבוה (כמעט 44%) למרות
שהמדדים הרפואיים שלה תקינים. גורמי הסיכון שלה הם הגיל ומספר ההיריונות.

</div>

<div dir="rtl" align="right">

---
# 3. ניתוח מעמיק — פעילות גופנית נמוכה מול גבוהה

בסקירה הכללית ראינו פער גדול בשיעור התחלואה בין הקבוצות. בחלק הזה אני
בודק אם הפער נובע באמת מהפעילות הגופנית, או ממשתנה אחר שמשפיע על שתיהן.

## 3.1 הגדרת הקבוצות

</div>

In [ ]:
low = df[df.PhysicalActivity == "low"]
high = df[df.PhysicalActivity == "high"]

print("פעילות נמוכה:", len(low), "מטופלות | אחוז חולות:",
      round(low.IsDiabetic.mean() * 100, 1))
print("פעילות גבוהה:", len(high), "מטופלות | אחוז חולות:",
      round(high.IsDiabetic.mean() * 100, 1))
print("הפרש:", round((low.IsDiabetic.mean() - high.IsDiabetic.mean()) * 100, 1),
      "נקודות אחוז")

<div dir="rtl" align="right">

## 3.2 השוואת המדדים בין הקבוצות

בנוסף להפרש בין הממוצעים חישבתי גם את גודל האפקט (Cohen's d), שמראה
כמה גדול ההבדל ביחס לפיזור של הנתונים. ערך מעל 0.8 נחשב אפקט גדול.

</div>

In [ ]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1 - 1) * group1.var() + (n2 - 1) * group2.var())
                         / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

compare = pd.DataFrame({
    "פעילות_נמוכה": low[num_cols].mean().round(2),
    "פעילות_גבוהה": high[num_cols].mean().round(2),
})
compare["הפרש"] = (compare["פעילות_גבוהה"] - compare["פעילות_נמוכה"]).round(2)
compare["cohens_d"] = [round(cohens_d(high[c], low[c]), 3) for c in num_cols]

compare.reindex(compare.cohens_d.abs().sort_values(ascending=False).index)

<div dir="rtl" align="right">

התוצאה מפתיעה: גודל האפקט הגדול ביותר הוא דווקא של `BMI` (1.93),
הרבה יותר מזה של הגלוקוז (0.29). כלומר ההבדל העיקרי בין שתי הקבוצות
הוא לא ברמת הסוכר אלא במשקל — 36.7 מול 27.0.

</div>

<div dir="rtl" align="right">

## 3.3 בדיקת הקשר בין המשקל לפעילות

אם קבוצת הפעילות הנמוכה שמנה בהרבה, ומשקל גבוה הוא בעצמו גורם סיכון
לסוכרת, ייתכן שהפער שייחסנו לפעילות נובע למעשה מהמשקל.

</div>

In [ ]:
two_groups = df[df.PhysicalActivity.isin(["low", "high"])]

print("התפלגות קטגוריות BMI בכל קבוצה (%):")
print(pd.crosstab(two_groups.BMICategory, two_groups.PhysicalActivity,
                  normalize="columns").mul(100).round(1))

activity_num = df.PhysicalActivity.map({"low": 0, "moderate": 1, "high": 2})
print("\nקורלציה בין BMI לרמת הפעילות:", round(df.BMI.corr(activity_num), 3))
print("קורלציה בין BMI לאבחון:", round(df.BMI.corr(df.IsDiabetic), 3))

<div dir="rtl" align="right">

שתי הקבוצות שונות מאוד בהרכב שלהן: בקבוצת הפעילות הנמוכה 90.6%
מוגדרות כסובלות מהשמנה ואין אף אחת במשקל תקין, לעומת 26.2% בקבוצת
הפעילות הגבוהה.

הקורלציה בין BMI לפעילות היא ‎-0.55‎, ובין BMI לאבחון 0.33. כלומר
המשקל קשור גם למשתנה המסביר וגם למשתנה המוסבר.

</div>

<div dir="rtl" align="right">

## 3.4 השוואה בתוך קבוצות משקל

כדי לנטרל את השפעת המשקל, משווים בין פעילות נמוכה לגבוהה בנפרד בתוך
כל קטגוריית BMI.

</div>

In [ ]:
rates = two_groups.pivot_table(index="BMICategory", columns="PhysicalActivity",
                               values="IsDiabetic", aggfunc="mean",
                               observed=True).mul(100).round(1)
sizes = two_groups.pivot_table(index="BMICategory", columns="PhysicalActivity",
                               values="IsDiabetic", aggfunc="count",
                               observed=True)

table = rates.add_suffix("_אחוז").join(sizes.add_suffix("_כמות"))
table["הפרש"] = (table["low_אחוז"] - table["high_אחוז"]).round(1)
table

<div dir="rtl" align="right">

הפער מצטמצם משמעותית:

* בקטגוריית ההשמנה: 46.3% מול 35.8%, הפרש של 10.4 נקודות אחוז בלבד
* בקטגוריית העודף משקל: 25.8% מול 22.4%, הפרש של 3.4 נקודות אחוז

בקטגוריית המשקל התקין אי אפשר להשוות, כי אין אף מטופלת בפעילות נמוכה
עם משקל תקין.

</div>

<div dir="rtl" align="right">

## 3.5 חישוב ההפרש המתוקנן

מחשבים ממוצע משוקלל של ההפרשים בתוך קבוצות המשקל, לפי גודל כל קבוצה.
כך מקבלים את ההפרש שנשאר אחרי שהמשקל מנוטרל.

</div>

In [ ]:
valid = table.dropna(subset=["הפרש"]).copy()
valid["סהכ"] = valid["low_כמות"] + valid["high_כמות"]

crude = (low.IsDiabetic.mean() - high.IsDiabetic.mean()) * 100
adjusted = (valid["הפרש"] * valid["סהכ"]).sum() / valid["סהכ"].sum()

print("הפרש לפני התיקנון:", round(crude, 1), "נקודות אחוז")
print("הפרש אחרי התיקנון למשקל:", round(adjusted, 1), "נקודות אחוז")
print("אחוז מההפרש שמוסבר על ידי המשקל:", round((1 - adjusted / crude) * 100), "%")

<div dir="rtl" align="right">

כ-62% מההפרש המקורי מוסבר בהבדלי המשקל בין הקבוצות ולא בפעילות
הגופנית עצמה. עדיין נשאר הפרש של 8.8 נקודות אחוז שקשור לפעילות.

</div>

<div dir="rtl" align="right">

## 3.6 ויזואליזציה

</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

pd.Series({"low": low.IsDiabetic.mean() * 100,
           "high": high.IsDiabetic.mean() * 100}).plot(
    kind="bar", ax=axes[0], color=["#9e3b31", "#2f7d78"])
axes[0].set_title("Diabetic Rate (%)")

(pd.crosstab(two_groups.BMICategory, two_groups.PhysicalActivity,
             normalize="columns").mul(100)).plot(
    kind="bar", ax=axes[1], color=["#2f7d78", "#9e3b31"])
axes[1].set_title("BMI Distribution by Activity (%)")

rates.dropna().plot(kind="bar", ax=axes[2], color=["#2f7d78", "#9e3b31"])
axes[2].set_title("Diabetic Rate Within BMI Groups (%)")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=two_groups, x="BMI", y="GlucoseMgDl",
                hue="PhysicalActivity", style="Diagnosis", alpha=0.55, s=28,
                palette={"low": "#9e3b31", "high": "#2f7d78"}, ax=ax)
ax.set_title("BMI vs Glucose by Activity Level")
plt.tight_layout()
plt.show()

<div dir="rtl" align="right">

---
# 4. בדיקות סטטיסטיות

## 4.1 בדיקת הנחות

לפני מבחן t צריך לבדוק שתי הנחות: האם המשתנה מתפלג נורמלית, והאם
השונויות של שתי הקבוצות שוות.

</div>

In [ ]:
healthy = df[df.Diagnosis == "healthy"]
diabetic = df[df.Diagnosis == "diabetic"]

shapiro_test = stats.shapiro(df.GlucoseMgDl.sample(500, random_state=42))
print("Shapiro-Wilk (בדיקת נורמליות): p =", round(shapiro_test.pvalue, 8))

levene_test = stats.levene(healthy.GlucoseMgDl, diabetic.GlucoseMgDl)
print("Levene (בדיקת שוויון שונויות): p =", round(levene_test.pvalue, 8))

<div dir="rtl" align="right">

שתי ההנחות לא מתקיימות במלואן. עם זאת, גודל המדגם הוא 4,500 ולכן
לפי משפט הגבול המרכזי מבחן t עדיין תקף. מכיוון שהשונויות אינן שוות,
משתמשים בגרסת Welch של המבחן, שלא מניחה שוויון שונויות.

בנוסף אריץ לצד כל מבחן פרמטרי גם מבחן לא-פרמטרי, כדי לוודא שהמסקנה
לא תלויה בהנחות.

</div>

<div dir="rtl" align="right">

## 4.2 השערה 1 — הבדל ברמת הגלוקוז בין חולות לבריאות

* **H0:** ממוצע הגלוקוז זהה בשתי הקבוצות
* **H1:** ממוצע הגלוקוז שונה בין הקבוצות

</div>

In [ ]:
t_test = stats.ttest_ind(diabetic.GlucoseMgDl, healthy.GlucoseMgDl,
                         equal_var=False)
print("Welch t-test: t =", round(t_test.statistic, 3), "| p =", t_test.pvalue)
print("Cohen's d:", round(cohens_d(diabetic.GlucoseMgDl, healthy.GlucoseMgDl), 3))

mann = stats.mannwhitneyu(diabetic.GlucoseMgDl, healthy.GlucoseMgDl)
print("Mann-Whitney (מבחן לא-פרמטרי): p =", mann.pvalue)

print("\nממוצעים: חולות", round(diabetic.GlucoseMgDl.mean(), 1),
      "| בריאות", round(healthy.GlucoseMgDl.mean(), 1))

<div dir="rtl" align="right">

p קטן מ-0.05 ולכן דוחים את H0. גודל האפקט 1.17 מעיד על הבדל גדול.
שני המבחנים מגיעים לאותה מסקנה.

</div>

<div dir="rtl" align="right">

## 4.3 השערה 2 — קשר בין פעילות גופנית לאבחון

שני המשתנים קטגוריים ולכן משתמשים במבחן חי בריבוע.

* **H0:** אין קשר בין רמת הפעילות לאבחון
* **H1:** קיים קשר

</div>

In [ ]:
table_activity = pd.crosstab(df.PhysicalActivity, df.Diagnosis)
chi2, p_value, dof, expected = stats.chi2_contingency(table_activity)

n = table_activity.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table_activity.shape) - 1)))

print("Chi-square =", round(chi2, 2), "| p =", p_value, "| df =", dof)
print("Cramer's V (גודל אפקט):", round(cramers_v, 3))
print()
print(table_activity)

<div dir="rtl" align="right">

דוחים את H0 — קיים קשר מובהק. עם זאת, ערך Cramer's V הוא 0.18,
כלומר הקשר מובהק אבל חלש. זה מתאים למה שראינו בחלק 3.

</div>

<div dir="rtl" align="right">

## 4.4 השערה 3 — הבדל בין המרפאות

* **H0:** ממוצע הגלוקוז זהה בכל המרפאות
* **H1:** קיים הבדל בין המרפאות

</div>

In [ ]:
clinic_groups = [group.GlucoseMgDl.values for name, group in df.groupby("Clinic")]
anova = stats.f_oneway(*clinic_groups)
print("ANOVA: F =", round(anova.statistic, 3), "| p =", round(anova.pvalue, 4))

table_clinic = pd.crosstab(df.Clinic, df.Diagnosis)
chi2_clinic, p_clinic, dof_clinic, exp_clinic = stats.chi2_contingency(table_clinic)
print("Chi-square (אבחון לפי מרפאה): p =", round(p_clinic, 4))

<div dir="rtl" align="right">

בשני המבחנים p גדול מ-0.05, ולכן לא דוחים את H0. אין הבדל מובהק
בין המרפאות — לא ברמת הגלוקוז ולא בשיעור התחלואה.

תוצאה כזו חשובה לא פחות מתוצאה מובהקת: היא מראה שההבדלים שמצאנו
בעבודה אינם נובעים מכך שמרפאה מסוימת מטפלת באוכלוסייה שונה.

</div>

<div dir="rtl" align="right">

## 4.5 השערה 4 — הקשר בין פעילות לאבחון בתוך קבוצת ההשמנה

בחלק 3 ראינו שרוב הפער מוסבר במשקל. כאן בודקים אם ההפרש שנשאר
מובהק סטטיסטית. המבחן מורץ רק על המטופלות בקטגוריית ההשמנה, כך
שהמשקל מנוטרל.

* **H0:** בתוך קבוצת ההשמנה אין קשר בין פעילות לאבחון
* **H1:** קיים קשר גם אחרי נטרול המשקל

</div>

In [ ]:
obese = df[(df.BMICategory == "Obese") &
           (df.PhysicalActivity.isin(["low", "high"]))]

table_obese = pd.crosstab(obese.PhysicalActivity, obese.Diagnosis)
chi2_obese, p_obese, dof_obese, exp_obese = stats.chi2_contingency(table_obese)

print("Chi-square =", round(chi2_obese, 2), "| p =", round(p_obese, 5))
print()
print(table_obese)

<div dir="rtl" align="right">

p = 0.0019, קטן מ-0.05, ולכן דוחים את H0. ההפרש שנשאר אחרי נטרול
המשקל אינו מקרי — לפעילות הגופנית יש קשר לתחלואה גם בתוך קבוצת
המטופלות בעלות אותו מבנה גוף.

</div>

<div dir="rtl" align="right">

## 4.6 השערה 5 — הבדל בין מדידת הגלוקוז הראשונה למדידת המעקב

כאן מדובר באותן מטופלות שנמדדו פעמיים, ולכן משתמשים במבחן t מזווג.

* **H0:** אין הבדל בין שתי המדידות
* **H1:** קיים הבדל

</div>

In [ ]:
paired = stats.ttest_rel(df.GlucoseFollowUp, df.GlucoseMgDl)
print("Paired t-test: t =", round(paired.statistic, 3), "| p =", paired.pvalue)
print("הפרש ממוצע:", round(df.GlucoseDelta.mean(), 2), "יחידות")

wilcoxon = stats.wilcoxon(df.GlucoseFollowUp, df.GlucoseMgDl)
print("Wilcoxon (מבחן לא-פרמטרי): p =", wilcoxon.pvalue)

<div dir="rtl" align="right">

דוחים את H0. חשוב לשים לב שההפרש בפועל קטן — ירידה ממוצעת של
5.8 יחידות גלוקוז. במדגם של 4,500 רשומות גם הפרש קטן יוצא מובהק
סטטיסטית, ולכן צריך להסתכל גם על גודל ההפרש ולא רק על ערך p.

</div>

<div dir="rtl" align="right">

## 4.7 השערה 6 — מתאם בין BMI לרמת הגלוקוז

* **H0:** מקדם המתאם שווה לאפס
* **H1:** קיים מתאם

</div>

In [ ]:
r, p_corr = stats.pearsonr(df.BMI, df.GlucoseMgDl)
print("Pearson: r =", round(r, 3), "| p =", p_corr)

r_spearman, p_spearman = stats.spearmanr(df.BMI, df.GlucoseMgDl)
print("Spearman: r =", round(r_spearman, 3))

<div dir="rtl" align="right">

דוחים את H0. קיים מתאם חיובי מובהק, אך מתון בעוצמתו (0.22).

</div>

<div dir="rtl" align="right">

## 4.8 סיכום הבדיקות

</div>

In [ ]:
summary = pd.DataFrame([
    ["גלוקוז: חולות מול בריאות", "Welch t-test", "3.1e-228", "נדחתה", "d = 1.17"],
    ["פעילות גופנית מול אבחון", "Chi-square", "1.3e-32", "נדחתה", "V = 0.18"],
    ["גלוקוז בין המרפאות", "ANOVA", "0.589", "לא נדחתה", "-"],
    ["אבחון בין המרפאות", "Chi-square", "0.905", "לא נדחתה", "-"],
    ["פעילות בתוך קבוצת ההשמנה", "Chi-square", "0.0019", "נדחתה", "V = 0.07"],
    ["מדידה ראשונה מול מעקב", "Paired t-test", "< 0.001", "נדחתה", "5.8 יחידות"],
    ["מתאם BMI וגלוקוז", "Pearson", "9.8e-51", "נדחתה", "r = 0.22"],
], columns=["השערה", "מבחן", "p", "H0", "גודל אפקט"])

summary

<div dir="rtl" align="right">

---
# 5. סיכום ומסקנות

## המדדים הקשורים לתחלואה

הגורמים בעלי הקשר החזק ביותר לאבחון סוכרת הם רמת הגלוקוז וה-HbA1c
(קורלציה של כ-0.49 ו-0.48), ואחריהם האינסולין וה-BMI. ההבדל בין
החולות לבריאות ברמת הגלוקוז נמצא מובהק סטטיסטית עם גודל אפקט גדול.

## שלוש קבוצות סיכון

חלוקת K-Means לשלוש קבוצות זיהתה חלוקה ברורה לפי רמת סיכון: קבוצה
צעירה עם 14% תחלואה, קבוצה עם עומס מטבולי גבוה ו-73% תחלואה, וביניהן
קבוצה של מטופלות מבוגרות יותר עם ריבוי היריונות שבה אחוז התחלואה
מגיע ל-44% למרות שהמדדים הרפואיים שלהן תקינים.

הקבוצה האמצעית מדגימה שסינון שמתבסס רק על בדיקות דם עלול לפספס
אוכלוסייה בסיכון, ושכדאי להתחשב גם בגיל ובמספר ההיריונות.

## פעילות גופנית ומשקל

בהשוואה ראשונית נראה שהפער בין פעילות נמוכה לגבוהה עצום — 44% מול 21%.
הניתוח המעמיק הראה שכ-62% מהפער הזה מוסבר למעשה בהבדלי משקל בין
הקבוצות, ולא בפעילות עצמה.

עם זאת, גם אחרי נטרול המשקל נשאר הפרש של 8.8 נקודות אחוז, והמבחן
הסטטיסטי שהורץ בתוך קבוצת ההשמנה בלבד אישר שההפרש הזה מובהק.
המסקנה: המשקל הוא הגורם החזק יותר, אך גם לפעילות הגופנית יש תרומה
עצמאית.

## ממצאים שליליים

לא נמצא הבדל מובהק בין חמש המרפאות ולא נמצאה מגמה לאורך שלוש השנים.
המשמעות היא שההבדלים בעבודה נובעים ממאפייני המטופלות עצמן ולא
מהמרפאה שבה טופלו או מהתקופה.

## מגבלות העבודה

* כל הרשומות בדאטהסט הן של נשים, ולכן המסקנות אינן ניתנות להכללה
  לגברים.
* בקבוצת הפעילות הנמוכה אין אף מטופלת במשקל תקין, ולכן לא ניתן
  לבודד את השפעת הפעילות בטווח המשקל הזה.
* הנתונים הם חתך רוחב בנקודת זמן ולא מעקב לאורך זמן, ולכן אפשר
  להצביע על קשרים אך לא להוכיח סיבתיות.

</div>

In [ ]:
df.to_csv("diabetes_final_output.csv", index=False, encoding="utf-8-sig")
print("הקובץ נשמר. מספר שורות:", len(df), "| מספר עמודות:", df.shape[1])